# 08 — CWE-95 Generalization Study (Eval Injection)

**Purpose.** Replicate the two central findings of the paper — *Instruction Overload* and *Answer Leakage / hint-leakage* — on a third, mechanistically distinct vulnerability class, **CWE-95 (Eval Injection)**, to test whether they generalize beyond CWE-78 and CWE-89.

This is an **independent test set** (it does not modify the original 234-sample benchmark). It runs four prompt conditions on GPT-4o-mini:

- `Variant_A_Baseline` — unstructured baseline
- `Variant_C_Patterns_CLEAN` — Patterns prompt, category-agnostic
- `Variant_E_Full_CLEAN` — Full framework, category-agnostic
- `Variant_E_Full_HINTED_CWE95` — Full framework, hinting **only CWE-95**

Together these support: *Instruction Overload* (E-CLEAN vs Baseline) and *hint-leakage* (E-HINTED vs E-CLEAN).

**Inputs.**
- `<BASE_DIR>/final_dataset_cwe95/` — produced by the CWE-95 curation step (96 vulnerable + 48 safe = 144 samples).
- `prompts/variant_A_baseline.txt`, `prompts/variant_C_patterns_clean.txt`, `prompts/variant_E_full_clean.txt`, `prompts/system_message.txt` — committed to the repository.
- A new CWE-95 HINTED prompt, defined inline in this notebook.
- An OpenAI API key with access to `gpt-4o-mini`.

**Outputs.**
- `<BASE_DIR>/results/cwe95_generalization_results.csv` — one row per sample, one column per condition.

**Cost & runtime.** 144 samples x 4 conditions = 576 API calls on gpt-4o-mini. Approx cost: USD 0.05-0.10. Approx runtime: 15-25 min. Resume-on-interrupt is supported.


## 1. Setup

On Google Colab, set `BASE_DIR_OVERRIDE` to your Drive path (the same one used for curation), e.g. `/content/drive/MyDrive/LLM_Security_Paper`. The OpenAI API key is read from `OPENAI_API_KEY`, falling back to an interactive prompt (never stored).

In [1]:
import os
import json
import time
from pathlib import Path

import pandas as pd
from tqdm.auto import tqdm
from openai import OpenAI

# ---- Mount Google Drive (Colab). Safe to re-run; skips if already mounted. ----
try:
    from google.colab import drive
    drive.mount('/content/drive')
    IN_COLAB = True
except Exception:
    IN_COLAB = False
    print('Not on Colab (or mount skipped); assuming local paths.')

# ---- USER-EDITABLE ----
BASE_DIR = Path('/content/drive/MyDrive/LLM_Security_Paper')  # parent of final_dataset_cwe95/
MODEL_ID          = 'gpt-4o-mini'
TEMPERATURE       = 0.1
MAX_RETRIES       = 3
RETRY_BACKOFF_SEC = 2
# -----------------------

DATASET_DIR = BASE_DIR / 'final_dataset_cwe95'
RESULTS_DIR = BASE_DIR / 'results'
OUTPUT_CSV  = RESULTS_DIR / 'cwe95_generalization_results.csv'
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

api_key = os.environ.get('OPENAI_API_KEY')
if not api_key:
    from getpass import getpass
    api_key = getpass('Enter your OpenAI API key (will not be stored): ')
client = OpenAI(api_key=api_key)

print(f'BASE_DIR    : {BASE_DIR}')
print(f'DATASET_DIR : {DATASET_DIR}  (exists: {DATASET_DIR.exists()})')
print(f'OUTPUT_CSV  : {OUTPUT_CSV}')
print(f'MODEL_ID    : {MODEL_ID}')
if not DATASET_DIR.exists():
    print('\n[WARNING] DATASET_DIR not found. Check that BASE_DIR points to your Drive project '
          'folder and that final_dataset_cwe95/ exists there (created by the curation step).')


Mounted at /content/drive
Enter your OpenAI API key (will not be stored): ··········
BASE_DIR    : /content/drive/MyDrive/LLM_Security_Paper
DATASET_DIR : /content/drive/MyDrive/LLM_Security_Paper/final_dataset_cwe95  (exists: True)
OUTPUT_CSV  : /content/drive/MyDrive/LLM_Security_Paper/results/cwe95_generalization_results.csv
MODEL_ID    : gpt-4o-mini


## 2. Load prompts

Four conditions. The Baseline (A), Patterns-CLEAN (C), and Full-CLEAN (E) prompts are read from the repository's `prompts/` directory (identical files used for the main study). The CWE-95 HINTED prompt is defined inline, mirroring `variant_E_full_hinted.txt` but naming **only CWE-95**.

**Category-neutral system message.** The original study's system message illustrates the `cwe_type` field with the examples `"CWE-89", "CWE-78"` — the two classes of the original benchmark. Carrying those examples into a CWE-95 evaluation would leak the original categories and undermine the *category-agnostic* (CLEAN) conditions. We therefore use a **category-neutral** system message here (examples replaced by the generic placeholder `"CWE-XX"`), defined inline below. This keeps the CLEAN conditions genuinely category-agnostic for the CWE-95 test set. This deviation from the original system message — adopted specifically to remove example leakage — is noted in the manuscript.


In [2]:
# All prompts defined inline so this notebook is fully self-contained
# (no dependency on a cloned repo / prompts/ directory).
# Contents are verbatim copies of the committed prompt files used in the main study,
# except the system message, which is made category-neutral (see Section 2 above).

PROMPTS = {
    # Variant A — unstructured baseline (verbatim from variant_A_baseline.txt)
    'Variant_A_Baseline':
        'Analyze the following PHP code. Does it contain any security '
        'vulnerabilities? Return JSON format.',

    # Variant C — Patterns, CLEAN / category-agnostic (verbatim from variant_C_patterns_clean.txt)
    'Variant_C_Patterns_CLEAN':
        'Analyze this PHP code. Apply taint analysis to identify data flow from '
        'untrusted user inputs (sources such as $_GET, $_POST, $_REQUEST, $_COOKIE) '
        'to security-sensitive operations (sinks). Determine whether any unsanitized '
        'tainted data reaches a sink. Return JSON.',

    # Variant E — Full framework, CLEAN / category-agnostic (verbatim from variant_E_full_clean.txt)
    'Variant_E_Full_CLEAN':
        'You are a senior cybersecurity auditor. Think step-by-step: '
        '1. Trace untrusted inputs (sources). 2. Identify security-sensitive '
        'operations (sinks). 3. Evaluate whether sanitization between source and '
        'sink is adequate. Return your final verdict in JSON format.',

    # Variant E — Full framework, HINTED on CWE-95 only
    # (mirrors variant_E_full_hinted.txt structure; "CWE-89 and CWE-78" -> "CWE-95")
    'Variant_E_Full_HINTED_CWE95':
        'You are a senior cybersecurity auditor. Think step-by-step: '
        '1. Trace inputs. 2. Look for CWE-95 patterns. Evaluate sanitization. '
        'Return your final verdict in JSON format.',
}

# Category-NEUTRAL system message (examples "CWE-89","CWE-78" -> generic "CWE-XX").
SYSTEM_MESSAGE = (
    'You must respond strictly in JSON format with exactly three keys:\n'
    '1. "prediction": String, MUST be either "Vulnerable" or "Safe".\n'
    '2. "cwe_type": String, the identified CWE (e.g., "CWE-XX") or "None" if safe.\n'
    '3. "confidence": Integer from 0 to 100.'
)

print('--- Prompts (inline) ---')
for name, text in PROMPTS.items():
    print(f'{name:30s} ({len(text):3d} chars): {text[:70]}{"..." if len(text) > 70 else ""}')
print()
print('--- Category-neutral system message ---')
print(SYSTEM_MESSAGE)


--- Prompts (inline) ---
Variant_A_Baseline             ( 97 chars): Analyze the following PHP code. Does it contain any security vulnerabi...
Variant_C_Patterns_CLEAN       (263 chars): Analyze this PHP code. Apply taint analysis to identify data flow from...
Variant_E_Full_CLEAN           (259 chars): You are a senior cybersecurity auditor. Think step-by-step: 1. Trace u...
Variant_E_Full_HINTED_CWE95    (170 chars): You are a senior cybersecurity auditor. Think step-by-step: 1. Trace i...

--- Category-neutral system message ---
You must respond strictly in JSON format with exactly three keys:
1. "prediction": String, MUST be either "Vulnerable" or "Safe".
2. "cwe_type": String, the identified CWE (e.g., "CWE-XX") or "None" if safe.
3. "confidence": Integer from 0 to 100.


## 3. Run the four conditions

The label of each sample is inferred from its parent directory under `final_dataset_cwe95/`:
`CWE_95_EvalInj` -> Vulnerable (CWE-95); `Safe_Code` -> Safe.
Output is written incrementally (safe to interrupt and resume). Per-call failures are retried up to `MAX_RETRIES`; persistent failures are recorded as `Error`.

In [3]:
def get_true_label(filepath: Path) -> tuple:
    """Infer (true_label, true_cwe) from the parent directory name (CWE-95 dataset)."""
    folder = filepath.parent.name
    if 'Safe' in folder:
        return 'Safe', None
    if '95' in folder:
        return 'Vulnerable', 'CWE-95'
    raise ValueError(f'Cannot infer label from folder name: {folder}')

def predict(prompt_text: str, code_content: str) -> str:
    for attempt in range(MAX_RETRIES):
        try:
            response = client.chat.completions.create(
                model=MODEL_ID,
                response_format={'type': 'json_object'},
                temperature=TEMPERATURE,
                messages=[
                    {'role': 'system', 'content': SYSTEM_MESSAGE},
                    {'role': 'user',   'content': f'{prompt_text}\n\nTarget Code:\n{code_content}'},
                ],
            )
            result = json.loads(response.choices[0].message.content)
            return result.get('prediction', 'Error')
        except Exception:
            if attempt + 1 < MAX_RETRIES:
                time.sleep(RETRY_BACKOFF_SEC)
            else:
                return 'Error'

all_files = sorted(DATASET_DIR.rglob('*.php'))
if not all_files:
    raise RuntimeError(f'No .php files found under {DATASET_DIR}. Run the CWE-95 curation step first.')
print(f'Total samples: {len(all_files)}  (expected 144)')

if OUTPUT_CSV.exists():
    results_df = pd.read_csv(OUTPUT_CSV)
    processed = set(results_df['File_Name'].tolist())
    print(f'Resuming: {len(processed)} samples already processed.')
else:
    columns = ['File_Name', 'True_Label', 'True_CWE'] + list(PROMPTS.keys())
    results_df = pd.DataFrame(columns=columns)
    processed = set()

to_process = [p for p in all_files if p.name not in processed]
print(f'Samples remaining: {len(to_process)}')

for filepath in tqdm(to_process, desc=f'CWE-95 study on {MODEL_ID}'):
    true_label, true_cwe = get_true_label(filepath)
    code_content = filepath.read_text(encoding='utf-8', errors='ignore')
    row = {'File_Name': filepath.name, 'True_Label': true_label, 'True_CWE': true_cwe}
    for variant_name, prompt_text in PROMPTS.items():
        row[variant_name] = predict(prompt_text, code_content)
    results_df = pd.concat([results_df, pd.DataFrame([row])], ignore_index=True)
    results_df.to_csv(OUTPUT_CSV, index=False, encoding='utf-8-sig')

print(f'\nComplete. Results written to {OUTPUT_CSV}')


Total samples: 144  (expected 144)
Samples remaining: 144


CWE-95 study on gpt-4o-mini:   0%|          | 0/144 [00:00<?, ?it/s]


Complete. Results written to /content/drive/MyDrive/LLM_Security_Paper/results/cwe95_generalization_results.csv


## 4. Inspect results

Sanity checks only. Full metric computation (Accuracy, F1, Recall, Specificity, Precision) and the two paired McNemar tests are done in the next notebook (`09_cwe95_stats.ipynb`).

In [4]:
df = pd.read_csv(OUTPUT_CSV)
print(f'Total rows : {len(df)}   (expected 144)')
print()
print('True label distribution:')
print(df['True_Label'].value_counts().to_string())
print()
print('Per-condition prediction distribution:')
for variant in PROMPTS.keys():
    counts = df[variant].value_counts().to_dict()
    print(f'  {variant:30s} {counts}')


Total rows : 144   (expected 144)

True label distribution:
True_Label
Vulnerable    96
Safe          48

Per-condition prediction distribution:
  Variant_A_Baseline             {'Vulnerable': 144}
  Variant_C_Patterns_CLEAN       {'Vulnerable': 144}
  Variant_E_Full_CLEAN           {'Vulnerable': 144}
  Variant_E_Full_HINTED_CWE95    {'Vulnerable': 144}


In [5]:
# 抓一個 SAFE 樣本，直接看模型的原始 JSON 回應
safe_files = sorted((DATASET_DIR / 'Safe_Code').glob('*.php'))
print(f'Safe 樣本數: {len(safe_files)}')

test_file = safe_files[0]
code_content = test_file.read_text(encoding='utf-8', errors='ignore')
print(f'\n測試檔案: {test_file.name}')
print(f'--- 程式碼前 500 字 ---')
print(code_content[:500])

# 用 E_CLEAN prompt 跑一次，印出完整原始回應
prompt_text = PROMPTS['Variant_E_Full_CLEAN']
response = client.chat.completions.create(
    model=MODEL_ID,
    response_format={'type': 'json_object'},
    temperature=TEMPERATURE,
    messages=[
        {'role': 'system', 'content': SYSTEM_MESSAGE},
        {'role': 'user',   'content': f'{prompt_text}\n\nTarget Code:\n{code_content}'},
    ],
)
raw = response.choices[0].message.content
print(f'\n--- 模型原始回應 (raw JSON) ---')
print(raw)

import json
parsed = json.loads(raw)
print(f'\n--- 解析後 ---')
print(f"prediction = {parsed.get('prediction')}")
print(f"cwe_type   = {parsed.get('cwe_type')}")
print(f"confidence = {parsed.get('confidence')}")

Safe 樣本數: 48

測試檔案: CWE_95__GET__CAST-cast_float__variable-concatenation_simple_quote.php
--- 程式碼前 500 字 ---
<?php
/* 
Safe sample
input : reads the field UserData from the variable $_GET
sanitize : cast in float
construction : concatenation with simple quote
*/



/*Copyright 2015 Bertrand STIVALET 

Permission is hereby granted, without written agreement or royalty fee, to

use, copy, modify, and distribute this software and its documentation for

any purpose, provided that the above copyright notice and the following

three paragraphs appear in all copies of this software.


IN NO EVENT SHALL AUTHOR

--- 模型原始回應 (raw JSON) ---
{
  "prediction": "Vulnerable",
  "cwe_type": "CWE-94",
  "confidence": 85
}

--- 解析後 ---
prediction = Vulnerable
cwe_type   = CWE-94
confidence = 85


In [6]:
import collections, json

# ========== Part 1: 用現有 CSV 能確認的 ==========
df = pd.read_csv(OUTPUT_CSV)
print('===== Part 1: 現有結果 CSV =====')
print(f'總列數: {len(df)}')
# 確認四組是否真的「逐筆完全相同」
cols = ['Variant_A_Baseline','Variant_C_Patterns_CLEAN','Variant_E_Full_CLEAN','Variant_E_Full_HINTED_CWE95']
all_same = (df[cols].nunique(axis=1) == 1).all()
print(f'四組逐筆完全相同嗎: {all_same}')
print(f'有任何一筆四組不一致嗎: {(df[cols].nunique(axis=1) > 1).sum()} 筆')
print()

# ========== Part 2: 重抓樣本看 cwe_type 分布 ==========
# 對 vulnerable 和 safe 各抽幾個，看模型回傳的 cwe_type 到底認成什麼
def get_raw(prompt_text, code_content):
    r = client.chat.completions.create(
        model=MODEL_ID, response_format={'type':'json_object'}, temperature=TEMPERATURE,
        messages=[{'role':'system','content':SYSTEM_MESSAGE},
                  {'role':'user','content':f'{prompt_text}\n\nTarget Code:\n{code_content}'}])
    return json.loads(r.choices[0].message.content)

vuln_files = sorted((DATASET_DIR / 'CWE_95_EvalInj').glob('*.php'))
safe_files = sorted((DATASET_DIR / 'Safe_Code').glob('*.php'))

print('===== Part 2: 模型對 cwe_type 的判斷（E_CLEAN，各抽 8 個）=====')
for label, files in [('VULN', vuln_files[:8]), ('SAFE', safe_files[:8])]:
    print(f'\n--- {label} 樣本 ---')
    for f in files:
        code = f.read_text(encoding='utf-8', errors='ignore')
        p = get_raw(PROMPTS['Variant_E_Full_CLEAN'], code)
        # 從檔名取 sanitize 方式（第3段）幫助判讀
        san = f.name.split('__')[2] if len(f.name.split('__'))>2 else '?'
        print(f"  pred={p.get('prediction'):11s} cwe={str(p.get('cwe_type')):10s} conf={p.get('confidence')}  [{san}]")

# ========== Part 3: HINTED vs CLEAN 在「同一個 safe 樣本」上有沒有差 ==========
print('\n===== Part 3: 同一 safe 樣本，HINTED vs CLEAN 比較（抽 5 個）=====')
for f in safe_files[:5]:
    code = f.read_text(encoding='utf-8', errors='ignore')
    pc = get_raw(PROMPTS['Variant_E_Full_CLEAN'], code)
    ph = get_raw(PROMPTS['Variant_E_Full_HINTED_CWE95'], code)
    print(f"  {f.name[:45]:45s}  CLEAN={pc.get('prediction'):11s}  HINTED={ph.get('prediction')}")

===== Part 1: 現有結果 CSV =====
總列數: 144
四組逐筆完全相同嗎: True
有任何一筆四組不一致嗎: 0 筆

===== Part 2: 模型對 cwe_type 的判斷（E_CLEAN，各抽 8 個）=====

--- VULN 樣本 ---
  pred=Vulnerable  cwe=CWE-94     conf=95  [no_sanitizing]
  pred=Vulnerable  cwe=CWE-94     conf=95  [no_sanitizing]
  pred=Vulnerable  cwe=CWE-94     conf=95  [no_sanitizing]
  pred=Vulnerable  cwe=CWE-94     conf=95  [no_sanitizing]
  pred=Vulnerable  cwe=CWE-94     conf=95  [no_sanitizing]
  pred=Vulnerable  cwe=CWE-94     conf=85  [no_sanitizing]
  pred=Vulnerable  cwe=CWE-94     conf=95  [no_sanitizing]
  pred=Vulnerable  cwe=CWE-94     conf=95  [no_sanitizing]

--- SAFE 樣本 ---
  pred=Vulnerable  cwe=CWE-94     conf=85  [CAST-cast_float]
  pred=Vulnerable  cwe=CWE-94     conf=85  [CAST-cast_float]
  pred=Vulnerable  cwe=CWE-94     conf=85  [CAST-cast_float]
  pred=Vulnerable  cwe=CWE-94     conf=85  [CAST-cast_float]
  pred=Vulnerable  cwe=CWE-94     conf=85  [CAST-cast_float]
  pred=Vulnerable  cwe=CWE-94     conf=85  [CAST-cast_float]
  pr

In [7]:
import collections, json

# ---- 從 raw clone 抽少量 CWE-98 樣本做探針（不複製檔案，直接讀）----
RAW_DIR_98 = Path('/content/drive/MyDrive/LLM_Security_Paper/raw_sard_php')

unsafe98 = sorted(p for p in RAW_DIR_98.rglob('*.php')
                  if p.name.startswith('CWE_98__') and '/unsafe/' in p.as_posix())
safe98   = sorted(p for p in RAW_DIR_98.rglob('*.php')
                  if p.name.startswith('CWE_98__') and '/safe/' in p.as_posix())

# 只取 no_sanitizing 的 unsafe（跟 CWE-95 策展邏輯一致，標籤無爭議）
vuln98 = [p for p in unsafe98 if p.name.split('__')[2] == 'no_sanitizing']
print(f'CWE-98 no_sanitizing vulnerable 候選: {len(vuln98)}')
print(f'CWE-98 safe 候選: {len(safe98)}')

# 看 unsafe 的注入技法分布（確認 no_sanitizing 存在）
tech98 = collections.Counter(p.name.split('__')[2] for p in unsafe98 if len(p.name.split('__'))>2)
print('CWE-98 unsafe 技法分布:', dict(tech98))
print()

def get_raw(prompt_text, code_content):
    r = client.chat.completions.create(
        model=MODEL_ID, response_format={'type':'json_object'}, temperature=TEMPERATURE,
        messages=[{'role':'system','content':SYSTEM_MESSAGE},
                  {'role':'user','content':f'{prompt_text}\n\nTarget Code:\n{code_content}'}])
    return json.loads(r.choices[0].message.content)

# ---- 探針：10 vuln + 10 safe，用 E-CLEAN ----
print('===== CWE-98 探針 (E_CLEAN) =====')
print('\n--- VULN 樣本 (應為 Vulnerable) ---')
vpreds = []
for f in vuln98[:10]:
    code = f.read_text(encoding='utf-8', errors='ignore')
    p = get_raw(PROMPTS['Variant_E_Full_CLEAN'], code)
    vpreds.append(p.get('prediction'))
    print(f"  pred={p.get('prediction'):11s} cwe={str(p.get('cwe_type')):10s} conf={p.get('confidence')}")

print('\n--- SAFE 樣本 (應為 Safe) ---')
spreds = []
for f in safe98[:10]:
    code = f.read_text(encoding='utf-8', errors='ignore')
    p = get_raw(PROMPTS['Variant_E_Full_CLEAN'], code)
    spreds.append(p.get('prediction'))
    san = f.name.split('__')[2] if len(f.name.split('__'))>2 else '?'
    print(f"  pred={p.get('prediction'):11s} cwe={str(p.get('cwe_type')):10s} conf={p.get('confidence')}  [{san}]")

# ---- 快速判讀 ----
print('\n===== 探針判讀 =====')
print(f'VULN: {vpreds.count("Vulnerable")}/10 判 Vulnerable (越高越好)')
print(f'SAFE: {spreds.count("Safe")}/10 判 Safe (越高越好；若 0 代表 specificity 又崩)')
cwes = collections.Counter()
print(f'是否全部 Vulnerable: {set(vpreds+spreds) == {"Vulnerable"}}')

CWE-98 no_sanitizing vulnerable 候選: 192
CWE-98 safe 候選: 2592
CWE-98 unsafe 技法分布: {'func_FILTER-CLEANING-email_filter': 96, 'func_FILTER-CLEANING-full_special_chars_filter': 96, 'func_FILTER-CLEANING-special_chars_filter': 96, 'func_FILTER-VALIDATION-email_filter': 96, 'func_preg_match-no_filtering': 96, 'no_sanitizing': 192}

===== CWE-98 探針 (E_CLEAN) =====

--- VULN 樣本 (應為 Vulnerable) ---
  pred=Vulnerable  cwe=CWE-94     conf=95
  pred=Vulnerable  cwe=CWE-98     conf=95
  pred=Vulnerable  cwe=CWE-20     conf=85
  pred=Vulnerable  cwe=CWE-98     conf=95
  pred=Vulnerable  cwe=CWE-98     conf=95
  pred=Vulnerable  cwe=CWE-94     conf=85
  pred=Vulnerable  cwe=CWE-74     conf=95
  pred=Vulnerable  cwe=CWE-94     conf=95
  pred=Vulnerable  cwe=CWE-94     conf=85
  pred=Vulnerable  cwe=CWE-98     conf=85

--- SAFE 樣本 (應為 Safe) ---
  pred=Vulnerable  cwe=CWE-94     conf=85  [CAST-cast_float]
  pred=Vulnerable  cwe=CWE-94     conf=85  [CAST-cast_float]
  pred=Vulnerable  cwe=CWE-20     conf

In [9]:
import collections, json

PROBE_MODEL = 'gpt-5.5-2026-04-23'

RAW = Path('/content/drive/MyDrive/LLM_Security_Paper/raw_sard_php')
unsafe98 = sorted(p for p in RAW.rglob('*.php')
                  if p.name.startswith('CWE_98__') and '/unsafe/' in p.as_posix())
safe98   = sorted(p for p in RAW.rglob('*.php')
                  if p.name.startswith('CWE_98__') and '/safe/' in p.as_posix())
vuln98 = [p for p in unsafe98 if p.name.split('__')[2] == 'no_sanitizing']

def get_raw(model, prompt_text, code):
    kwargs = dict(
        model=model,
        response_format={'type':'json_object'},
        messages=[{'role':'system','content':SYSTEM_MESSAGE},
                  {'role':'user','content':f'{prompt_text}\n\nTarget Code:\n{code}'}],
    )
    # GPT-5.5 只接受預設 temperature(=1)，不送此參數；其他模型維持 0.1
    if not model.startswith('gpt-5'):
        kwargs['temperature'] = TEMPERATURE
    r = client.chat.completions.create(**kwargs)
    return json.loads(r.choices[0].message.content)

print(f'===== CWE-98 探針 ({PROBE_MODEL}, E_CLEAN) =====')
print('\n--- VULN 樣本 (應為 Vulnerable) ---')
vpreds = []
for f in vuln98[:10]:
    p = get_raw(PROBE_MODEL, PROMPTS['Variant_E_Full_CLEAN'], f.read_text(encoding='utf-8', errors='ignore'))
    vpreds.append(p.get('prediction'))
    print(f"  pred={str(p.get('prediction')):11s} cwe={str(p.get('cwe_type')):10s} conf={p.get('confidence')}")

print('\n--- SAFE 樣本 (應為 Safe) ---')
spreds = []
for f in safe98[:10]:
    p = get_raw(PROBE_MODEL, PROMPTS['Variant_E_Full_CLEAN'], f.read_text(encoding='utf-8', errors='ignore'))
    spreds.append(p.get('prediction'))
    san = f.name.split('__')[2] if len(f.name.split('__'))>2 else '?'
    print(f"  pred={str(p.get('prediction')):11s} cwe={str(p.get('cwe_type')):10s} conf={p.get('confidence')}  [{san}]")

print('\n===== 判讀 =====')
print(f'VULN: {vpreds.count("Vulnerable")}/10 判 Vulnerable')
print(f'SAFE: {spreds.count("Safe")}/10 判 Safe   ← 關鍵：>0 代表有區辨力')
print(f'是否全部 Vulnerable: {set(vpreds+spreds) == {"Vulnerable"}}')

===== CWE-98 探針 (gpt-5.5-2026-04-23, E_CLEAN) =====

--- VULN 樣本 (應為 Vulnerable) ---
  pred=Vulnerable  cwe=CWE-98     conf=99
  pred=Vulnerable  cwe=CWE-98     conf=96
  pred=Vulnerable  cwe=CWE-98     conf=82
  pred=Vulnerable  cwe=CWE-98     conf=95
  pred=Vulnerable  cwe=CWE-98     conf=95
  pred=Vulnerable  cwe=CWE-98     conf=95
  pred=Vulnerable  cwe=CWE-98     conf=96
  pred=Vulnerable  cwe=CWE-98     conf=98
  pred=Vulnerable  cwe=CWE-98     conf=82
  pred=Vulnerable  cwe=CWE-98     conf=98

--- SAFE 樣本 (應為 Safe) ---
  pred=Safe        cwe=None       conf=95  [CAST-cast_float]
  pred=Safe        cwe=None       conf=95  [CAST-cast_float]
  pred=Safe        cwe=None       conf=95  [CAST-cast_float]
  pred=Safe        cwe=None       conf=95  [CAST-cast_float]
  pred=Safe        cwe=None       conf=95  [CAST-cast_float]
  pred=Safe        cwe=None       conf=95  [CAST-cast_float]
  pred=Safe        cwe=None       conf=95  [CAST-cast_float_sort_of]
  pred=Safe        cwe=None      

In [10]:
import shutil, collections
from pathlib import Path

RAW = Path('/content/drive/MyDrive/LLM_Security_Paper/raw_sard_php')
OUT = Path('/content/drive/MyDrive/LLM_Security_Paper/final_dataset_cwe98')
VULN_DIR = OUT / 'CWE_98_FileIncl'
SAFE_DIR = OUT / 'Safe_Code'
for d in (VULN_DIR, SAFE_DIR):
    d.mkdir(parents=True, exist_ok=True)

# 1. Vulnerable: 取 96 個 no_sanitizing（CWE-98 有 192 個，取前 96 對齊 CWE-95 規模）
all_unsafe = sorted(p for p in RAW.rglob('*.php')
                    if p.name.startswith('CWE_98__') and '/unsafe/' in p.as_posix())
vuln_all = [p for p in all_unsafe if p.name.split('__')[2] == 'no_sanitizing']
print(f'no_sanitizing 總候選: {len(vuln_all)}')

# 分層：依 source 均勻取，湊到 96
by_src_v = collections.defaultdict(list)
for p in vuln_all:
    by_src_v[p.name.split('__')[1]].append(p)
n_src = len(by_src_v)
per_src_v = 96 // n_src
vuln = []
for src in sorted(by_src_v):
    vuln.extend(sorted(by_src_v[src])[:per_src_v])
# 若因整除有缺，從剩餘補足到 96
if len(vuln) < 96:
    remaining = [p for p in vuln_all if p not in set(vuln)]
    vuln.extend(sorted(remaining)[:96-len(vuln)])
vuln = vuln[:96]
print(f'選取 vulnerable: {len(vuln)}  (每 source 約 {per_src_v} 個，共 {n_src} 種 source)')

# 2. Safe: 分層抽樣，依 source 均勻取 48
all_safe = sorted(p for p in RAW.rglob('*.php')
                  if p.name.startswith('CWE_98__') and '/safe/' in p.as_posix())
by_src_s = collections.defaultdict(list)
for p in all_safe:
    by_src_s[p.name.split('__')[1]].append(p)
per_src_s = 48 // len(by_src_s)
safe_sel = []
for src in sorted(by_src_s):
    safe_sel.extend(sorted(by_src_s[src])[:per_src_s])
if len(safe_sel) < 48:
    remaining = [p for p in all_safe if p not in set(safe_sel)]
    safe_sel.extend(sorted(remaining)[:48-len(safe_sel)])
safe_sel = safe_sel[:48]
print(f'選取 safe: {len(safe_sel)}')

# 3. 複製
for p in vuln:     shutil.copy2(p, VULN_DIR / p.name)
for p in safe_sel: shutil.copy2(p, SAFE_DIR / p.name)

# 4. file list
file_list = ([f'CWE_98_FileIncl/{p.name}' for p in vuln] +
             [f'Safe_Code/{p.name}' for p in safe_sel])
with open(OUT / 'file_list_cwe98.txt', 'w') as f:
    f.write('\n'.join(sorted(file_list)) + '\n')

# 5. 驗證
print('\n========== 策展完成 ==========')
print(f'Vulnerable: {len(list(VULN_DIR.glob("*.php")))}')
print(f'Safe:       {len(list(SAFE_DIR.glob("*.php")))}')
print(f'合計:       {len(list(VULN_DIR.glob("*.php"))) + len(list(SAFE_DIR.glob("*.php")))}')
print(f'輸出: {OUT}')

no_sanitizing 總候選: 192
選取 vulnerable: 96  (每 source 約 6 個，共 16 種 source)
選取 safe: 48

========== 策展完成 ==========
Vulnerable: 96
Safe:       48
合計:       144
輸出: /content/drive/MyDrive/LLM_Security_Paper/final_dataset_cwe98
